  Using cached rapidfuzz-3.14.6-cp313-cp313-win_amd64.whl.metadata (12 kB)
Using cached rapidfuzz-3.14.6-cp313-cp313-win_amd64.whl (1.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("data/spotify_tracks_final.csv")
df["_title"] = df["_title"].fillna("").astype(str)
df["_artist"] = df["_artist"].fillna("").astype(str)
X = np.load("data/X_scaled.npy")
assert len(df) == len(X)

knn = NearestNeighbors(metric="euclidean").fit(X)


def find_song(title, artist=None):
    pool = df
    if artist:
        pool = df[df["_artist"].str.contains(artist.lower().strip(), regex=False)]
    if pool.empty:
        return None
    hit = process.extractOne(title.lower().strip(), pool["_title"],
                             scorer=fuzz.WRatio, score_cutoff=85)
    if hit is None:
        return None
    rows = pool[pool["_title"] == hit[0]]
    return rows["popularity"].idxmax()


def recommend(title, artist=None, n=5, min_pop=50):
    pos = find_song(title, artist)
    if pos is None:
        return None
    q = df.loc[pos]
    genres = q["all_genres"].split(", ")

    mask = (df["track_genre"].isin(genres)
            & (df["popularity"] >= min_pop)
            & (df.index != pos)
            & (df["_artist"] != q["_artist"]))
    sub = df[mask].copy()
    if sub.empty:
        return None

    d = np.linalg.norm(X[sub.index] - X[pos], axis=1)
    sub["audio_sim"] = 1 / (1 + d)
    sub["score"] = sub["audio_sim"] + 0.2 * sub["popularity"] / 100

    sub = sub.sort_values("score", ascending=False)
    sub = sub.groupby("_artist", sort=False).head(1)
    return sub.head(n)[["track_name", "artists", "track_genre", "score"]]


for song, artist in [("Blinding Lights", "The Weeknd"),
                     ("Shape of You", "Ed Sheeran"),
                     ("Bohemian Rhapsody", "Queen")]:
    print(f"\n=== {song} ===")
    out = recommend(song, artist)
    print("Not found" if out is None else out.to_string(index=False))


=== Blinding Lights ===
               track_name                     artists track_genre    score
STAY (with Justin Bieber) The Kid LAROI;Justin Bieber         pop 0.747071
                    Ghost               Justin Bieber         pop 0.606180
           Piya O Re Piya   Atif Aslam;Shreya Ghoshal         pop 0.593253
                  Choo Lo             The Local Train         pop 0.588702
           Wildest Dreams                Taylor Swift         pop 0.580999

=== Shape of You ===
                                    track_name                            artists track_genre    score
               There's Nothing Holdin' Me Back                       Shawn Mendes         pop 0.607606
                     Naah Goriye (From "Bala") B Praak;Harrdy Sandhu;Swasti Mehul         pop 0.604830
                                          Naah                      Harrdy Sandhu         pop 0.587804
       Left and Right (Feat. Jung Kook of BTS)         Charlie Puth;Jung Kook;BTS         p